<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

# ?Por qu? utilizar Random Forest para este problema?

El modelo Random Forest es un algoritmo de ensemble basado en m?ltiples ?rboles de decisi?n que combina las predicciones de distintos ?rboles para generar un modelo m?s robusto y estable.

En este proyecto, el objetivo es clasificar el riesgo Urban Heat Island (UHI) (`No Risk` / `Risk`) utilizando variables espectrales, ambientales y topogr?ficas obtenidas a partir de im?genes satelitales y datos geogr?ficos.

Random Forest resulta especialmente adecuado para este caso porque:

- Maneja correctamente relaciones no lineales entre variables ambientales.
- Es robusto frente al ruido y variabilidad de los datos satelitales.
- Reduce el riesgo de sobreajuste al combinar m?ltiples ?rboles de decisi?n.
- Funciona bien con datasets tabulares de tama?o medio y gran cantidad de variables predictoras.
- Permite analizar la importancia de variables para interpretar qu? factores influyen m?s en el riesgo t?rmico urbano.

Adem?s, Random Forest tolera relativamente bien correlaciones entre variables y no requiere supuestos estad?sticos estrictos sobre la distribuci?n de los datos.

A diferencia de modelos como SVM, Random Forest no necesita escalado de variables, ya que basa su funcionamiento en particiones realizadas mediante ?rboles de decisi?n y no en distancias matem?ticas.

Finalmente, se evaluar? el impacto de incluir o eliminar las coordenadas geogr?ficas (`latitude`, `longitude`) para analizar hasta qu? punto el modelo depende de patrones espaciales expl?citos frente a variables ambientales derivadas.

</div>

In [ ]:
# ============================================================
# IMPORTACI?N DE LIBRER?AS
# ============================================================

# Manipulaci?n y an?lisis de datos
import numpy as np
import pandas as pd

# Modelo XGBoost
import xgboost as xgb

# Divisi?n del dataset y validaci?n cruzada
from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold
)

# Modelos de Machine Learning
from sklearn.ensemble import RandomForestClassifier

# M?tricas de evaluaci?n
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)

# Visualizaci?n de datos
import matplotlib.pyplot as plt
import seaborn as sns

# Configuraci?n general
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# CARGA DEL DATASET DE MODELADO
# ============================================================

# Cargamos el dataset final preparado durante el EDA
# y el proceso de feature engineering.

df_bcn = pd.read_csv('../data/processed/dataset_modeling.csv')
df_bcn.shape

In [ ]:
# ============================================================
# DEFINICI?N DE FEATURES Y TARGET
# ============================================================

# Seleccionamos las variables predictoras que utilizaremos
# para entrenar el modelo Random Forest.

features_rf = [
    'latitude', 'longitude', 'elevation',
    'mndwi', 'nbr', 'ndbi', 'ndmi',
    'ndvi', 'ndwi', 'nir',
    'swir1', 'swir2',
    'season'
]

# Definimos el dataset de entrada (X)
# y la variable objetivo (y).

X_rf = df_bcn[features_rf]

y_rf = df_bcn['uhi_risk']

In [ ]:
# ============================================================
# CODIFICACI?N DE VARIABLES CATEG?RICAS
# ============================================================

# Aplicamos One-Hot Encoding sobre la variable 'season'
# para evitar introducir relaciones ordinales artificiales.

X_rf = pd.get_dummies(
    X_rf,
    columns=['season'],
    drop_first=True
)

# Actualizamos la lista de variables tras la codificaci?n.

updated_features_rf = X_rf.columns.tolist()

# Verificamos las columnas generadas.

print(updated_features_rf)

In [ ]:
# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

# Dividimos el dataset en entrenamiento y prueba
# manteniendo el balance de clases.

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_rf,
    y_rf,
    test_size=0.2,
    random_state=42,
    stratify=y_rf
)

print(f"Dimensiones entrenamiento: {X_train_rf.shape[0]}")
print(f"Dimensiones prueba: {X_test_rf.shape[0]}")

In [ ]:
# ============================================================
# ENTRENAMIENTO RANDOM FOREST
# ============================================================

# Entrenamos el modelo Random Forest utilizando
# las variables espectrales, ambientales y espaciales.

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_rf, y_train_rf)

In [ ]:
# ============================================================
# EVALUACI?N RANDOM FOREST
# ============================================================

# Evaluamos el rendimiento del modelo sobre
# el conjunto de prueba.

y_pred_rf = rf_model.predict(X_test_rf)

print("=" * 60)
print("CLASSIFICATION REPORT ? RANDOM FOREST CON COORDENADAS")
print("=" * 60)

print(
    classification_report(
        y_test_rf,
        y_pred_rf,
        target_names=['No Risk', 'Risk']
    )
)

In [ ]:
# ============================================================
# MATRIZ DE CONFUSI?N RANDOM FOREST
# ============================================================

# Construimos la matriz de confusi?n para analizar
# los aciertos y errores de clasificaci?n del modelo.

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test_rf,
    y_pred_rf,
    display_labels=['No Risk', 'Risk'],
    cmap='Blues',
    ax=ax
)

ax.set_title('Matriz de Confusi?n ? Random Forest')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# IMPORTANCIA DE VARIABLES RANDOM FOREST
# ============================================================

# Analizamos la importancia de variables utilizando
# el criterio Gini del modelo Random Forest.

importances_rf = pd.Series(
    rf_model.feature_importances_,
    index=updated_features_rf
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))

importances_rf.plot(
    kind='barh',
    ax=ax,
    color='steelblue'
)

ax.set_title('Importancia de Variables (Gini) ? Random Forest')
ax.set_xlabel('Importancia')
ax.set_ylabel('Variable')

plt.tight_layout()
plt.show()

<div style="background-color:#fff3cd; padding:20px; border-left:8px solid #ffcc00; border-radius:8px; color:#1b1b1b;">

## **Observaci?n**

El modelo baseline de Random Forest muestra un rendimiento inicial consistente utilizando variables espectrales, ambientales y espaciales. Sin embargo, la importancia de variables sugiere que las coordenadas geogr?ficas (`latitude` y `longitude`) podr?an estar influyendo significativamente en las decisiones del modelo.

Dado que Random Forest est? basado en m?ltiples ?rboles de decisi?n, existe el riesgo de que parte del aprendizaje se apoye excesivamente en patrones espaciales espec?ficos de Barcelona en lugar de relaciones ambientales generalizables.

Por este motivo, antes de considerar el modelo como candidato final, resulta relevante entrenar una segunda versi?n eliminando las coordenadas geogr?ficas. Esto permitir? analizar si el modelo mantiene capacidad predictiva utilizando ?nicamente variables derivadas de im?genes satelitales, ?ndices espectrales y variables topogr?ficas.

Esta comparaci?n ayudar? a evaluar el grado de dependencia espacial del modelo y su capacidad de generalizaci?n frente a nuevos escenarios geogr?ficos.

</div>

In [ ]:
# ============================================================
# DEFINICI?N DE FEATURES Y TARGET SIN COORDENADAS
# ============================================================

# Seleccionamos las variables predictoras eliminando
# latitude y longitude para analizar la dependencia espacial.

features_rf_no_geo = [
    'elevation',
    'mndwi', 'nbr', 'ndbi', 'ndmi',
    'ndvi', 'ndwi', 'nir',
    'swir1', 'swir2',
    'season'
]

# Definimos el dataset de entrada (X)
# y la variable objetivo (y).

X_rf_no_geo = df_bcn[features_rf_no_geo]

y_rf_no_geo = df_bcn['uhi_risk']

In [ ]:
# ============================================================
# CODIFICACI?N DE VARIABLES CATEG?RICAS
# ============================================================

# Aplicamos One-Hot Encoding sobre la variable 'season'
# para evitar introducir relaciones ordinales artificiales.

X_rf_no_geo = pd.get_dummies(
    X_rf_no_geo,
    columns=['season'],
    drop_first=True
)

# Actualizamos la lista de variables tras la codificaci?n.

updated_features_rf_no_geo = X_rf_no_geo.columns.tolist()

# Verificamos las columnas generadas.

print(updated_features_rf_no_geo)

In [ ]:
# ============================================================
# TRAIN / TEST SPLIT SIN COORDENADAS
# ============================================================

# Dividimos el dataset en entrenamiento y prueba
# manteniendo el balance de clases.

X_train_rf_no_geo, X_test_rf_no_geo, y_train_rf_no_geo, y_test_rf_no_geo = train_test_split(
    X_rf_no_geo,
    y_rf_no_geo,
    test_size=0.2,
    random_state=42,
    stratify=y_rf_no_geo
)

print(f"Dimensiones entrenamiento: {X_train_rf_no_geo.shape[0]}")
print(f"Dimensiones prueba: {X_test_rf_no_geo.shape[0]}")

In [ ]:
# ============================================================
# ENTRENAMIENTO RANDOM FOREST SIN COORDENADAS
# ============================================================

# Entrenamos el modelo Random Forest utilizando
# ?nicamente variables espectrales, ambientales
# y topogr?ficas.

rf_model_no_geo = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model_no_geo.fit(
    X_train_rf_no_geo,
    y_train_rf_no_geo
)

In [ ]:
# ============================================================
# EVALUACI?N RANDOM FOREST SIN COORDENADAS
# ============================================================

# Evaluamos el rendimiento del modelo sobre
# el conjunto de prueba.

y_pred_rf_no_geo = rf_model_no_geo.predict(X_test_rf_no_geo)

print("=" * 60)
print("CLASSIFICATION REPORT ? RANDOM FOREST SIN COORDENADAS")
print("=" * 60)

print(
    classification_report(
        y_test_rf_no_geo,
        y_pred_rf_no_geo,
        target_names=['No Risk', 'Risk']
    )
)

<div style="background-color:#d5f5e3; padding:20px; border-left:8px solid #27AE60; border-radius:8px; color:#1b1b1b;">

## **Observaci?n ? Selecci?n del modelo final**

El modelo Random Forest mostr? un rendimiento s?lido para la clasificaci?n del riesgo Urban Heat Island (UHI), especialmente cuando se utilizaron coordenadas geogr?ficas (`latitude` y `longitude`), alcanzando una accuracy aproximada de `0.71` y un recall de `0.73` para la clase `Risk`.

Sin embargo, al eliminar las coordenadas geogr?ficas, el rendimiento del modelo disminuy? de forma moderada (`accuracy ? 0.67`, `recall ? 0.71`), lo que sugiere una dependencia parcial de la informaci?n espacial para mantener su capacidad predictiva.

A pesar de ello, el modelo continu? identificando patrones relevantes utilizando ?nicamente variables espectrales, ambientales y topogr?ficas, lo que demuestra cierta capacidad de generalizaci?n m?s all? de la localizaci?n exacta de los puntos geogr?ficos.

Durante la comparaci?n global entre modelos, XGBoost present? un mejor equilibrio entre:
- capacidad predictiva,
- robustez espacial,
- estabilidad de m?tricas,
- y comportamiento general sin coordenadas geogr?ficas.

Adem?s, XGBoost logr? mantener un rendimiento m?s consistente al eliminar `latitude` y `longitude`, indicando una menor dependencia espacial y una mejor capacidad para aprender relaciones ambientales generalizables.

Por este motivo, aunque Random Forest fue utilizado como modelo de comparaci?n y validaci?n adicional dentro del an?lisis, el modelo finalmente seleccionado para el despliegue en Streamlit fue el modelo XGBoost optimizado sin coordenadas.

</div>